# Feature Engineering and Syntactic Similarity

## Setup<div class='tocSkip'/>

Set directory locations. If working on Google Colab: copy files and install required libraries.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load Python Settings<div class="tocSkip"/>

Common imports, defaults for formatting in Matplotlib, Pandas etc.

In [15]:
# suppress warnings
import warnings;
warnings.filterwarnings('ignore');

# common imports
import pandas as pd
import numpy as np
import math
import re
import glob
import os
import sys
import json
import random
import pprint as pp
import textwrap
import sqlite3
import logging

import spacy
import nltk

from tqdm.auto import tqdm
# register `pandas.progress_apply` and `pandas.Series.map_apply` with `tqdm`
tqdm.pandas()

# pandas display options
# https://pandas.pydata.org/pandas-docs/stable/user_guide/options.html#available-options
pd.options.display.max_columns = 30 # default 20
pd.options.display.max_rows = 60 # default 60
pd.options.display.float_format = '{:.2f}'.format
# pd.options.display.precision = 2
pd.options.display.max_colwidth = 200 # default 50; -1 = all
# otherwise text between $ signs will be interpreted as formula and printed in italic
pd.set_option('display.html.use_mathjax', False)

# np.set_printoptions(edgeitems=3) # default 3

import matplotlib
from matplotlib import pyplot as plt

plot_params = {'figure.figsize': (8, 4), 
               'axes.labelsize': 'large',
               'axes.titlesize': 'large',
               'xtick.labelsize': 'large',
               'ytick.labelsize':'large',
               'figure.dpi': 100}
# adjust matplotlib defaults
matplotlib.rcParams.update(plot_params)

import seaborn as sns
sns.set_style("darkgrid")


In [16]:
BASE_DIR = '/content/drive/MyDrive/ESCP/NLP/03_Text_Vectorization_Classification'
os.chdir(BASE_DIR)

# Data preparation

In [17]:
sentences = ["It was the best of times", 
             "it was the worst of times", 
             "it was the age of wisdom", 
             "it was the age of foolishness"]

tokenized_sentences = [[t for t in sentence.split()] for sentence in sentences]

vocabulary = set([w for s in tokenized_sentences for w in s])

In [18]:
tokenized_sentences

[['It', 'was', 'the', 'best', 'of', 'times'],
 ['it', 'was', 'the', 'worst', 'of', 'times'],
 ['it', 'was', 'the', 'age', 'of', 'wisdom'],
 ['it', 'was', 'the', 'age', 'of', 'foolishness']]

In [19]:
vocabulary

{'It',
 'age',
 'best',
 'foolishness',
 'it',
 'of',
 'the',
 'times',
 'was',
 'wisdom',
 'worst'}

In [20]:
vocab = {i:w for i,w in enumerate(vocabulary)}
vocab

{0: 'of',
 1: 'foolishness',
 2: 'times',
 3: 'the',
 4: 'best',
 5: 'age',
 6: 'wisdom',
 7: 'It',
 8: 'worst',
 9: 'was',
 10: 'it'}

In [21]:
import pandas as pd
[[w, i] for i,w in enumerate(vocabulary)]

[['of', 0],
 ['foolishness', 1],
 ['times', 2],
 ['the', 3],
 ['best', 4],
 ['age', 5],
 ['wisdom', 6],
 ['It', 7],
 ['worst', 8],
 ['was', 9],
 ['it', 10]]

# One-hot by hand

In [22]:
def onehot_encode(tokenized_sentence):
    return [1 if w in tokenized_sentence else 0 for w in vocabulary]

In [23]:
onehot = [onehot_encode(tokenized_sentence) for tokenized_sentence in tokenized_sentences]

In [24]:
onehot

[[1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0],
 [1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1],
 [1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1],
 [1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1]]

In [25]:
for (sentence, oh) in zip(sentences, onehot):
    print("%s: %s" % (oh, sentence))

[1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0]: It was the best of times
[1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1]: it was the worst of times
[1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1]: it was the age of wisdom
[1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1]: it was the age of foolishness


In [26]:
pd.DataFrame(onehot, index= sentences, columns=vocabulary)

,of,foolishness,times,the,best,age,wisdom,It,worst,was,it
It was the best of times,1,0,1,1,1,0,0,1,0,1,0
it was the worst of times,1,0,1,1,0,0,0,0,1,1,1
it was the age of wisdom,1,0,0,1,0,1,1,0,0,1,1
it was the age of foolishness,1,1,0,1,0,1,0,0,0,1,1


## document term matrix (DTM)
It is basically a Bag of Words with BINARY weighing

In [27]:
onehot

[[1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0],
 [1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1],
 [1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1],
 [1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1]]

### with CountVectorizer()

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
ohcv = CountVectorizer(binary=False)
ohcv.fit(sentences)

CountVectorizer()

In [29]:
pd.DataFrame(ohcv.transform(sentences).toarray(), index = sentences, columns = ohcv.get_feature_names_out())

,age,best,foolishness,it,of,the,times,was,wisdom,worst
It was the best of times,0,1,0,1,1,1,1,1,0,0
it was the worst of times,0,0,0,1,1,1,1,1,0,1
it was the age of wisdom,1,0,0,1,1,1,0,1,1,0
it was the age of foolishness,1,0,1,1,1,1,0,1,0,0


### with MultiLabelBinarizer()

In [30]:
from sklearn.preprocessing import MultiLabelBinarizer
lb = MultiLabelBinarizer()
lb.fit([vocabulary])
lb.transform(tokenized_sentences)

array([[1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0],
       [0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1],
       [0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0],
       [0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0]])

In [31]:
pd.DataFrame(lb.fit_transform(tokenized_sentences), index=sentences, columns=vocabulary)

,of,foolishness,times,the,best,age,wisdom,It,worst,was,it
It was the best of times,1,0,1,0,0,1,1,1,1,0,0
it was the worst of times,0,0,0,0,1,1,1,1,1,0,1
it was the age of wisdom,0,1,0,0,1,1,1,0,1,1,0
it was the age of foolishness,0,1,0,1,1,1,1,0,1,0,0


# Similarities

In [32]:
sim = [onehot[0][i] & onehot[1][i] for i in range(0, len(vocabulary))]
sum(sim)

4

In [33]:
import numpy as np
np.dot(onehot[0], onehot[1])

4

In [34]:
np.dot(onehot, onehot[1])

array([4, 6, 4, 4])

In [35]:
import numpy as np
np.dot(onehot, np.transpose(onehot))

array([[6, 4, 3, 3],
       [4, 6, 4, 4],
       [3, 4, 6, 5],
       [3, 4, 5, 6]])

## Out of vocabulary

In [36]:
onehot_encode("the age of wisdom is the best of times".split())

[1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0]

In [37]:
 onehot_encode("John likes to watch movies. Mary likes movies too.".split())

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

# CountVectorizer

In [38]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [39]:
more_sentences = sentences + ["John likes to watch movies. Mary likes movies too.",
                              "Mary also likes to watch football games."]
pd.DataFrame(more_sentences)

,0
0,It was the best of times
1,it was the worst of times
2,it was the age of wisdom
3,it was the age of foolishness
4,John likes to watch movies. Mary likes movies too.
5,Mary also likes to watch football games.


In [40]:
cv.fit(more_sentences)

CountVectorizer()

In [41]:
print(cv.get_feature_names_out())

['age' 'also' 'best' 'foolishness' 'football' 'games' 'it' 'john' 'likes'
 'mary' 'movies' 'of' 'the' 'times' 'to' 'too' 'was' 'watch' 'wisdom'
 'worst']


In [42]:
dt = cv.transform(more_sentences)

In [43]:
pd.DataFrame(dt.toarray(), index = more_sentences, columns=cv.get_feature_names_out())

,age,also,best,foolishness,football,games,it,john,likes,mary,movies,of,the,times,to,too,was,watch,wisdom,worst
It was the best of times,0,0,1,0,0,0,1,0,0,0,0,1,1,1,0,0,1,0,0,0
it was the worst of times,0,0,0,0,0,0,1,0,0,0,0,1,1,1,0,0,1,0,0,1
it was the age of wisdom,1,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,1,0,1,0
it was the age of foolishness,1,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,1,0,0,0
John likes to watch movies. Mary likes movies too.,0,0,0,0,0,0,0,1,2,1,2,0,0,0,1,1,0,1,0,0
Mary also likes to watch football games.,0,1,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,1,0,0


In [44]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity(dt[0], dt[1])

array([[0.83333333]])

In [45]:
?? cosine_similarity

In [46]:
A = dt[0].toarray()
B = dt[1].toarray()
np.dot(A,np.transpose(B))/(np.linalg.norm(A)*np.linalg.norm(B))

array([[0.83333333]])

In [47]:
len(more_sentences)

6

In [48]:
pd.DataFrame(cosine_similarity(dt, dt), index=more_sentences, columns=more_sentences)

,It was the best of times,it was the worst of times,it was the age of wisdom,it was the age of foolishness,John likes to watch movies. Mary likes movies too.,Mary also likes to watch football games.
It was the best of times,1.00,0.83,0.67,0.67,0.00,0.00
it was the worst of times,0.83,1.00,0.67,0.67,0.00,0.00
it was the age of wisdom,0.67,0.67,1.00,0.83,0.00,0.00
it was the age of foolishness,0.67,0.67,0.83,1.00,0.00,0.00
John likes to watch movies. Mary likes movies too.,0.00,0.00,0.00,0.00,1.00,0.52
Mary also likes to watch football games.,0.00,0.00,0.00,0.00,0.52,1.00


# TF/IDF

In [49]:
from sklearn.feature_extraction.text import TfidfTransformer
tfidf = TfidfTransformer()
tfidf_dt = tfidf.fit_transform(dt)

In [ ]:
pd.DataFrame(tfidf_dt.toarray(), index=more_sentences, columns=cv.get_feature_names_out(()))

,age,also,best,foolishness,football,games,it,john,likes,mary,movies,of,the,times,to,too,was,watch,wisdom,worst
It was the best of times,0.00,0.00,0.57,0.00,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.34,0.34,0.47,0.00,0.00,0.34,0.00,0.00,0.00
it was the worst of times,0.00,0.00,0.00,0.00,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.34,0.34,0.47,0.00,0.00,0.34,0.00,0.00,0.57
it was the age of wisdom,0.47,0.00,0.00,0.00,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.34,0.34,0.00,0.00,0.00,0.34,0.00,0.57,0.00
it was the age of foolishness,0.47,0.00,0.00,0.57,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.34,0.34,0.00,0.00,0.00,0.34,0.00,0.00,0.00
John likes to watch movies. Mary likes movies too.,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.31,0.50,0.25,0.61,0.00,0.00,0.00,0.25,0.31,0.00,0.25,0.00,0.00
Mary also likes to watch football games.,0.00,0.42,0.00,0.00,0.42,0.42,0.00,0.00,0.34,0.34,0.00,0.00,0.00,0.00,0.34,0.00,0.00,0.34,0.00,0.00


In [50]:
pd.DataFrame(cosine_similarity(tfidf_dt, tfidf_dt), index=more_sentences, columns=more_sentences)

,It was the best of times,it was the worst of times,it was the age of wisdom,it was the age of foolishness,John likes to watch movies. Mary likes movies too.,Mary also likes to watch football games.
It was the best of times,1.00,0.68,0.46,0.46,0.00,0.00
it was the worst of times,0.68,1.00,0.46,0.46,0.00,0.00
it was the age of wisdom,0.46,0.46,1.00,0.68,0.00,0.00
it was the age of foolishness,0.46,0.46,0.68,1.00,0.00,0.00
John likes to watch movies. Mary likes movies too.,0.00,0.00,0.00,0.00,1.00,0.43
Mary also likes to watch football games.,0.00,0.00,0.00,0.00,0.43,1.00


In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidfv = TfidfVectorizer()
tfidfv_dt = tfidfv.fit_transform(more_sentences)

In [53]:
pd.DataFrame(tfidfv_dt.toarray(), index=more_sentences, columns=tfidfv.get_feature_names_out())

,age,also,best,foolishness,football,games,it,john,likes,mary,movies,of,the,times,to,too,was,watch,wisdom,worst
It was the best of times,0.00,0.00,0.57,0.00,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.34,0.34,0.47,0.00,0.00,0.34,0.00,0.00,0.00
it was the worst of times,0.00,0.00,0.00,0.00,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.34,0.34,0.47,0.00,0.00,0.34,0.00,0.00,0.57
it was the age of wisdom,0.47,0.00,0.00,0.00,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.34,0.34,0.00,0.00,0.00,0.34,0.00,0.57,0.00
it was the age of foolishness,0.47,0.00,0.00,0.57,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.34,0.34,0.00,0.00,0.00,0.34,0.00,0.00,0.00
John likes to watch movies. Mary likes movies too.,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.31,0.50,0.25,0.61,0.00,0.00,0.00,0.25,0.31,0.00,0.25,0.00,0.00
Mary also likes to watch football games.,0.00,0.42,0.00,0.00,0.42,0.42,0.00,0.00,0.34,0.34,0.00,0.00,0.00,0.00,0.34,0.00,0.00,0.34,0.00,0.00


In [54]:
ABCNEWS_FILE = './data/abcnews-date-text.csv.gz'

In [55]:
headlines = pd.read_csv(ABCNEWS_FILE, parse_dates=["publish_date"])
headlines.head()

,publish_date,headline_text
0,2003-02-19,aba decides against community broadcasting licence
1,2003-02-19,act fire witnesses must be aware of defamation
2,2003-02-19,a g calls for infrastructure protection summit
3,2003-02-19,air nz staff in aust strike for pay rise
4,2003-02-19,air nz strike to affect australian travellers


In [57]:
headlines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1103663 entries, 0 to 1103662
Data columns (total 2 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   publish_date   1103663 non-null  datetime64[ns]
 1   headline_text  1103663 non-null  object        
dtypes: datetime64[ns](1), object(1)
memory usage: 16.8+ MB


In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
dt = tfidf.fit_transform(headlines["headline_text"])

In [58]:
dt

<1103663x95878 sparse matrix of type '<class 'numpy.float64'>'
	with 7001357 stored elements in Compressed Sparse Row format>

In [ ]:
dt.data.nbytes

56010856

In [ ]:
%%time
cosine_similarity(dt[0:10000], dt[0:10000])

CPU times: user 214 ms, sys: 652 ms, total: 865 ms
Wall time: 853 ms


array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.16913596,
        0.16792138],
       [0.        , 0.        , 0.        , ..., 0.16913596, 1.        ,
        0.33258708],
       [0.        , 0.        , 0.        , ..., 0.16792138, 0.33258708,
        1.        ]])

## Stopwords

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS as stopwords
print(len(stopwords))
tfidf = TfidfVectorizer(stop_words=list(stopwords))
dt = tfidf.fit_transform(headlines["headline_text"])
dt

326


<1103663x95600 sparse matrix of type '<class 'numpy.float64'>'
	with 5644186 stored elements in Compressed Sparse Row format>

## min_df

In [ ]:
tfidf = TfidfVectorizer(stop_words=list(stopwords), min_df=2)
dt = tfidf.fit_transform(headlines["headline_text"])
dt

<1103663x58527 sparse matrix of type '<class 'numpy.float64'>'
	with 5607113 stored elements in Compressed Sparse Row format>

In [ ]:
tfidf = TfidfVectorizer(stop_words=list(stopwords), min_df=.0001)
dt = tfidf.fit_transform(headlines["headline_text"])
dt

<1103663x6772 sparse matrix of type '<class 'numpy.float64'>'
	with 4816381 stored elements in Compressed Sparse Row format>

## max_df

In [ ]:
tfidf = TfidfVectorizer(stop_words=list(stopwords), max_df=0.1)
dt = tfidf.fit_transform(headlines["headline_text"])
dt

<1103663x95600 sparse matrix of type '<class 'numpy.float64'>'
	with 5644186 stored elements in Compressed Sparse Row format>

In [ ]:
tfidf = TfidfVectorizer(max_df=0.1)
dt = tfidf.fit_transform(headlines["headline_text"])
dt

<1103663x95875 sparse matrix of type '<class 'numpy.float64'>'
	with 6532752 stored elements in Compressed Sparse Row format>

## n-grams

In [ ]:
tfidf = TfidfVectorizer(stop_words=list(stopwords), ngram_range=(1,2), min_df=2)
dt = tfidf.fit_transform(headlines["headline_text"])
print(dt.shape)
print(dt.data.nbytes)

tfidf = TfidfVectorizer(stop_words=list(stopwords), ngram_range=(1,3), min_df=2)
dt = tfidf.fit_transform(headlines["headline_text"])
print(dt.shape)
print(dt.data.nbytes)

(1103663, 559961)
67325400
(1103663, 747988)
72360104


## Lemmas

In [ ]:
if spacy.prefer_gpu():
    print("Working on GPU.")
else:
    print("No GPU found, working on CPU.")

Working on GPU.


In [ ]:
from tqdm.auto import tqdm
import spacy
nlp = spacy.load("en_core_web_sm")
nouns_adjectives_verbs = ["NOUN", "PROPN", "ADJ", "ADV", "VERB"]
for i, row in tqdm(headlines.iterrows(), total=len(headlines)):
    doc = nlp(str(row["headline_text"]))
    headlines.at[i, "lemmas"] = " ".join([token.lemma_ for token in doc])
    headlines.at[i, "nav"] = " ".join([token.lemma_ for token in doc if token.pos_ in nouns_adjectives_verbs])

  0%|          | 0/1103663 [00:00<?, ?it/s]

In [ ]:
headlines.head()

In [ ]:
tfidf = TfidfVectorizer(stop_words=stopwords)
dt = tfidf.fit_transform(headlines["lemmas"].map(str))
dt

In [ ]:
tfidf = TfidfVectorizer(stop_words=stopwords)
dt = tfidf.fit_transform(headlines["nav"].map(str))
dt

## remove top 10,000

In [ ]:
top_10000 = pd.read_csv("https://raw.githubusercontent.com/first20hours/google-10000-english/master/google-10000-english.txt", header=None)
tfidf = TfidfVectorizer(stop_words=set(top_10000.iloc[:,0].values))
dt = tfidf.fit_transform(headlines["nav"].map(str))
dt

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2), stop_words=set(top_10000.iloc[:,0].values), min_df=2)
dt = tfidf.fit_transform(headlines["nav"].map(str))
dt